# Build Your First Voice Agent using Twilio

This cookbook shows how to build a **real-time voice agent that answers phone calls**, using **[Twilio](https://www.twilio.com)** for telephony, **[Pipecat](https://docs.pipecat.ai)** to orchestrate the real-time audio pipeline, and **[Sarvam AI](https://docs.sarvam.ai)** for speech-to-text, the LLM, and text-to-speech.

It's a great starting point for IVR replacements, customer support lines, and conversational phone agents in Indian languages.

By the end of this notebook you will have an agent that can:

- Answer inbound phone calls on a Twilio number
- Listen to callers speaking in multiple Indian languages
- Understand and process what they say
- Respond back in a natural-sounding voice, over the phone

> **Note on running this notebook:** A phone voice agent is a long-running WebSocket server that Twilio streams call audio to — it isn't something you can "run to completion" in a notebook cell. This notebook builds the agent file for you (`agent.py`) using `%%writefile`, and walks through the Twilio console configuration (TwiML Bin, ngrok tunnel) you need alongside it.

## Overview

| | |
|---|---|
| **Telephony** | [Twilio](https://console.twilio.com) (Media Streams over WebSocket) |
| **Pipeline orchestration** | [Pipecat](https://docs.pipecat.ai) |
| **Speech-to-text** | Sarvam **Saarika/Saaras** (`SarvamSTTService`) |
| **LLM** | Sarvam chat models (`SarvamLLMService`, default `sarvam-105b`) |
| **Text-to-speech** | Sarvam **Bulbul v3** (`SarvamTTSService`) |
| **Languages** | 10 Indian languages + English, with auto-detect |
| **Lines of code** | ~80 |

### Quick overview of the steps

1. Get API keys (Twilio, Sarvam).
2. Install `pipecat-ai` with the `websocket` and `sarvam` extras.
3. Create a `.env` file with your credentials.
4. Write the agent (`agent.py`).
5. Point a Twilio phone number at your agent using a TwiML Bin + ngrok.
6. Call your Twilio number.

## Prerequisites

- Python 3.9 or higher.
- [ngrok](https://ngrok.com/download), to expose your local server during development.
- A [Twilio](https://console.twilio.com) account: an **Account SID**, an **Auth Token**, and a phone number with voice capability.
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

## 1. Install dependencies

The `websocket` extra adds Pipecat's WebSocket/FastAPI transport (used to accept Twilio's Media Streams connection); the `sarvam` extra adds the Sarvam STT/LLM/TTS services.

In [ ]:
%pip install -Uqq "pipecat-ai[websocket,sarvam]" python-dotenv loguru

## 2. Configure your credentials

Get your credentials from the [Sarvam dashboard](https://dashboard.sarvam.ai) and the [Twilio Console](https://console.twilio.com). Never commit real API keys to version control — the cell below writes a `.env.example` template; copy it to `.env` and fill in your real values locally:

```bash
cp .env.example .env
```

> `TWILIO_ACCOUNT_SID` and `TWILIO_AUTH_TOKEN` aren't optional here — Pipecat's Twilio transport uses them to authenticate with Twilio's REST API and to hang up the call cleanly when your agent ends the conversation.

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

SARVAM_API_KEY=your_sarvam_api_key
TWILIO_ACCOUNT_SID=your_twilio_account_sid
TWILIO_AUTH_TOKEN=your_twilio_auth_token

## 3. Write the voice agent

Same Pipecat pipeline shape as a browser-based agent (transport in → STT → LLM → TTS → transport out), but the transport is Twilio's Media Streams over a WebSocket, and audio runs at Twilio's native 8kHz mono sample rate.

`%%writefile` saves this cell's contents to `agent.py` in the current directory.

In [ ]:
%%writefile agent.py
import os

from dotenv import load_dotenv
from loguru import logger
from pipecat.frames.frames import LLMRunFrame
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.runner import PipelineRunner
from pipecat.pipeline.task import PipelineParams, PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
)
from pipecat.runner.types import RunnerArguments
from pipecat.runner.utils import create_transport
from pipecat.services.sarvam.llm import SarvamLLMService
from pipecat.services.sarvam.stt import SarvamSTTService
from pipecat.services.sarvam.tts import SarvamTTSService
from pipecat.transports.websocket.fastapi import FastAPIWebsocketParams

load_dotenv(override=True)


async def bot(runner_args: RunnerArguments) -> None:
    """Main bot entry point."""

    # create_transport auto-detects Twilio's WebSocket handshake and builds the
    # matching TwilioFrameSerializer, using TWILIO_ACCOUNT_SID and
    # TWILIO_AUTH_TOKEN from the environment.
    transport = await create_transport(
        runner_args,
        {
            "twilio": lambda: FastAPIWebsocketParams(
                audio_in_enabled=True, audio_out_enabled=True
            ),
        },
    )

    # Initialize the Sarvam AI services.
    stt = SarvamSTTService(api_key=os.getenv("SARVAM_API_KEY"))
    tts = SarvamTTSService(api_key=os.getenv("SARVAM_API_KEY"))
    llm = SarvamLLMService(
        api_key=os.getenv("SARVAM_API_KEY"),
        settings=SarvamLLMService.Settings(model="sarvam-105b"),
    )

    # Set up conversation context.
    messages = [
        {
            "role": "system",
            "content": "You are a friendly AI phone assistant. Keep your responses brief and conversational.",
        },
    ]
    context = LLMContext(messages)
    context_aggregator = LLMContextAggregatorPair(context)

    # Build the pipeline: audio in -> STT -> LLM -> TTS -> audio out.
    pipeline = Pipeline(
        [
            transport.input(),
            stt,
            context_aggregator.user(),
            llm,
            tts,
            transport.output(),
            context_aggregator.assistant(),
        ]
    )

    # Twilio Media Streams send and receive 8kHz mono audio.
    task = PipelineTask(
        pipeline,
        params=PipelineParams(
            audio_in_sample_rate=8000,
            audio_out_sample_rate=8000,
        ),
    )

    @transport.event_handler("on_client_connected")
    async def on_client_connected(transport, client) -> None:
        logger.info("Caller connected")
        messages.append(
            {"role": "system", "content": "Greet the caller and briefly introduce yourself."}
        )
        await task.queue_frames([LLMRunFrame()])

    @transport.event_handler("on_client_disconnected")
    async def on_client_disconnected(transport, client) -> None:
        logger.info("Caller disconnected")
        await task.cancel()

    runner = PipelineRunner(handle_sigint=runner_args.handle_sigint)
    await runner.run(task)


if __name__ == "__main__":
    from pipecat.runner.run import main

    main()

## 4. Configure Twilio to reach your agent

Twilio needs a public `wss://` URL to stream call audio to. For local development, expose your agent with ngrok.

**Start ngrok** (in a separate terminal):

```bash
ngrok http 7860
```

ngrok prints a forwarding URL such as `https://xxxx.ngrok-free.app`. You'll turn this into a `wss://` URL below.

> The domain suffix ngrok assigns varies per account (`ngrok-free.app`, `ngrok-free.dev`, or the older `ngrok.io`) — always use whatever URL ngrok actually prints.
>
> Free ngrok URLs change every time you restart ngrok. If your agent stops receiving calls, check whether the URL in your TwiML Bin still matches the one ngrok is currently printing.

**Create a TwiML Bin:**

1. Open the [Twilio Console](https://console.twilio.com) and search for **TwiML Bins** using the search bar at the top (or find it under **Developer Tools**).
2. Click **Create new TwiML Bin**, give it a friendly name, and paste the XML below, replacing the domain with your own ngrok URL.
3. Click **Create** to save the bin.

**Assign it to your phone number:**

1. Go to **Numbers & Senders** → **Phone Numbers**, and select your number.
2. Open its configuration/voice settings tab.
3. Find the setting for what happens when **a call comes in**, and set it to use your TwiML Bin.
4. Save.

In [ ]:
%%writefile twiml_bin.xml
<?xml version="1.0" encoding="UTF-8"?>
<Response>
  <Connect>
    <Stream url="wss://xxxx.ngrok-free.app/ws" />
  </Connect>
</Response>

Replace `xxxx.ngrok-free.app` above with your actual ngrok forwarding host before pasting this into the Twilio Console's TwiML Bin editor.

## 5. Run and test your agent

```bash
python agent.py --transport twilio
```

This starts a local FastAPI server, via Pipecat's development runner, that accepts Twilio's Media Streams WebSocket connection at `/ws`.

Call your Twilio phone number from any phone — your voice agent will pick up and start the conversation.

## Customization examples

Swap these blocks into the service-initialization section of `agent.py` and re-run.

> `language`, `voice`, and `target_language_code` are `Settings` fields, not constructor keyword arguments. Passing them directly to `SarvamSTTService(...)` or `SarvamTTSService(...)` raises a `TypeError` on current versions of `pipecat-ai`. Only `mode` (STT) and `api_key` stay outside `settings`.

### Hindi voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="hi-IN"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3",
        voice="simran",  # or: priya, ishita, kavya, aditya, anand, rohan
        target_language_code="hi-IN",
    ),
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

### Tamil voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="ta-IN"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="shubh", target_language_code="ta-IN"
    ),
)
```

### Multilingual agent (auto-detect)

Use `language="unknown"` for support lines where callers might speak any of several languages. Sarvam auto-detects the spoken language per utterance, so the same agent can handle a Hindi caller followed by a Tamil caller without any code changes.

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="unknown"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="anand", target_language_code="en-IN"
    ),
)
```

### Speech-to-English agent (Saaras)

Saarika transcribes speech to text in the same language. Saaras translates speech directly to English text. Use Saaras when the caller speaks an Indian language but you want to process and respond in English — it auto-detects the source language (Hindi, Tamil, etc.) and translates spoken content directly to English text.

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3"),
    mode="translate",  # speech-to-English translation
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="aditya", target_language_code="en-IN"
    ),
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

## Available options

### Language codes

| Language | Code |
|---|---|
| English (India) | `en-IN` |
| Hindi | `hi-IN` |
| Bengali | `bn-IN` |
| Tamil | `ta-IN` |
| Telugu | `te-IN` |
| Gujarati | `gu-IN` |
| Kannada | `kn-IN` |
| Malayalam | `ml-IN` |
| Marathi | `mr-IN` |
| Punjabi | `pa-IN` |
| Odia | `od-IN` |
| Auto-detect | `unknown` |

### Speaker voices (Bulbul v3)

- **Male (23):** shubh (default), aditya, rahul, rohan, amit, dev, ratan, varun, manan, sumit, kabir, aayan, ashutosh, advait, anand, tarun, sunny, mani, gokul, vijay, mohit, rehan, soham
- **Female (14):** ritu, priya, neha, pooja, simran, kavya, ishita, shreya, roopa, tanya, shruti, suhani, kavitha, rupali

### TTS additional parameters

```python
tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3",
        voice="shubh",
        target_language_code="en-IN",
        pace=1.0,  # range: 0.5 to 2.0
    ),
)
```

> Twilio Media Streams always carry 8kHz mono audio. You don't need to touch `sample_rate` on `SarvamTTSService` for this — Pipecat resamples the TTS output to match `audio_out_sample_rate` (set in `PipelineParams`) automatically.

## Understanding the call flow

```mermaid
flowchart LR
    caller(["Caller"]) <--> twilio["Twilio"]
    twilio <-->|"wss:// audio"| pipeline

    subgraph pipeline["Your Agent Server (Pipecat)"]
        direction LR
        stt["STT"] --> llm["LLM"] --> tts["TTS"]
    end

    pipeline -.->|HTTPS| sarvam["Sarvam AI\nSaarika/Saaras · sarvam-30b/105b · Bulbul"]
```

1. **Caller dials your Twilio number.** Twilio answers using the TwiML Bin and opens a WebSocket (Media Stream) to your agent.
2. **Transport input** — your Pipecat transport receives the caller's audio over the WebSocket and hands it to STT.
3. **STT** — converts audio to text using Sarvam's Saarika or Saaras; the context aggregator adds it to the conversation context.
4. **LLM** — generates a response using Sarvam.
5. **TTS** — converts the response to audio using Sarvam's Bulbul.
6. **Transport output** — streams the audio back over the WebSocket, and Twilio plays it to the caller, while the context aggregator saves the assistant's response to context.

## Pro tips

- Use `language="unknown"` to automatically detect the caller's language — works well for multilingual support lines, but transcription accuracy is slightly lower than specifying the exact language code, so prefer an explicit code whenever you already know it.
- Sarvam's models understand code-mixing, so your agent can naturally handle Hinglish, Tanglish, and other mixed languages, which is common on real support calls.
- Use `sarvam-30b` for faster responses, or `sarvam-105b` for more complex conversations.
- Log the transcript from `context_aggregator` if you want post-call analytics — writing it to a file or database is cheap and won't slow down the live pipeline.

## Troubleshooting

- **Call connects but there's no audio** — Confirm the TwiML `<Stream url>` uses `wss://`, not `wss:/` or `https://`, and that it points at your current ngrok URL. Free ngrok URLs change on every restart.
- **Call drops immediately** — Check that `TWILIO_ACCOUNT_SID` and `TWILIO_AUTH_TOKEN` are set correctly in `.env`. The Twilio serializer needs both to manage the call.
- **API key errors** — Make sure all keys are in your `.env` file and that the file sits in the same directory as `agent.py`.
- **Module not found** — Re-run the install cell for your operating system.
- **Poor transcription** — Try `language="unknown"` for auto-detection, or set the exact language code (`en-IN`, `hi-IN`, etc.) if you already know it.
- **Choppy or robotic audio** — Make sure `audio_in_sample_rate` and `audio_out_sample_rate` are both set to `8000` in `PipelineParams`. Twilio Media Streams don't support other rates, and a mismatch causes distorted playback.

## Additional resources

- [Sarvam AI Documentation](https://docs.sarvam.ai)
- [Pipecat Documentation](https://docs.pipecat.ai)
- [Pipecat Sarvam LLM Service](https://docs.pipecat.ai/api-reference/server/services/llm/sarvam)
- [Pipecat Twilio Serializer Reference](https://reference-server.pipecat.ai/en/latest/api/pipecat.serializers.twilio.html)
- [Twilio Media Streams Documentation](https://www.twilio.com/docs/voice/twiml/stream)
- [Twilio Getting Started with TwiML Bins](https://www.twilio.com/docs/serverless/twiml-bins/getting-started)
- [Twilio Console](https://console.twilio.com)

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)